# Phase 3 (Stage 1) -- HeteroGNN training

Trains [`FraudHeteroGNN`](../src/fraud_detection/models/hetero_gnn.py) on the
Phase 2 heterogeneous graph (`data/graphs/hetero.pt`).

**Architecture (plan p. 7):** input projections per node type → 3× `HeteroGNNLayer`
(SAGEConv on tx↔entity, GATConv-4head on card↔card; residual + LayerNorm + ELU + Dropout 0.3)
→ 64-d embedding head → 32→1 classifier.

**Loss:** Focal (α=0.75, γ=2.0). **Opt:** AdamW(lr=1e-3, wd=1e-4) + cosine schedule.
**Early-stop:** patience=15 on val AUPRC.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    src = ROOT / 'src'
elif (ROOT.parent / 'src').exists():
    src = ROOT.parent / 'src'
else:
    raise RuntimeError('cannot find src/')
sys.path.insert(0, str(src))

import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.transforms import ToUndirected

from fraud_detection.utils.config import load_config
from fraud_detection.models import FraudHeteroGNN
from fraud_detection.training import (
    Trainer, TrainerConfig, ensure_temporal_masks,
    write_evaluation_report,
)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 110
cfg = load_config()
print(f'Project root: {cfg.paths.data_graphs.parents[1]}')

## 1. Load + prepare graph

Load `hetero.pt`, cast features to float32, add reverse edges with `ToUndirected`, and ensure the 60/20/20 chronological splits are set.

In [ ]:
data = torch.load(cfg.paths.data_graphs / 'hetero.pt', weights_only=False)
for nt in data.node_types:
    data[nt].x = data[nt].x.float()
data = ToUndirected()(data)
data = ensure_temporal_masks(data, target_node_type='transaction')

summary = pd.Series({
    'n_nodes': sum(data[nt].num_nodes for nt in data.node_types),
    'n_edges': sum(data[et].edge_index.shape[1] for et in data.edge_types),
    'n_node_types': len(data.node_types),
    'n_edge_types': len(data.edge_types),
    'train_n': int(data['transaction'].train_mask.sum()),
    'val_n': int(data['transaction'].val_mask.sum()),
    'test_n': int(data['transaction'].test_mask.sum()),
    'train_fraud_rate': float(data['transaction'].y[data['transaction'].train_mask].float().mean()),
    'val_fraud_rate': float(data['transaction'].y[data['transaction'].val_mask].float().mean()),
    'test_fraud_rate': float(data['transaction'].y[data['transaction'].test_mask].float().mean()),
}, name='value')
summary

## 2. Build the model and inspect it

In [ ]:
node_feature_dims = {nt: data[nt].num_node_features for nt in data.node_types}
model = FraudHeteroGNN(node_feature_dims=node_feature_dims, edge_types=data.edge_types)
print(model)
print(f'\nparams: {model.n_parameters():,}')

## 3. Train

Bump `epochs` to 100 for the production run. On a 50K subset / CPU each epoch is ~6 s.

In [ ]:
tcfg = TrainerConfig(
    epochs=30,                # 100 in prod
    early_stop_patience=10,   # 15 in prod
    log_every_n_epochs=2,
    mlflow_enabled=False,     # set True to log to mlruns/
)
trainer = Trainer(model, tcfg)
result = trainer.fit(data)
history = pd.DataFrame(result['history'])
history.tail()

## 4. Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['epoch'], history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['epoch'], history['val_auprc']); axes[1].set_title('val AUPRC'); axes[1].set_xlabel('epoch')
axes[2].plot(history['epoch'], history['lr']); axes[2].set_title('lr (cosine)'); axes[2].set_xlabel('epoch')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()

## 5. Val + test eval reports

In [ ]:
import numpy as np
model = result['model']
model.eval()
with torch.no_grad():
    logits_all = model(data)
probs = torch.sigmoid(logits_all).cpu().numpy()
y_all = data['transaction'].y.cpu().numpy().astype(np.int8)

for split in ('val', 'test'):
    mask = getattr(data['transaction'], f'{split}_mask').cpu().numpy()
    paths = write_evaluation_report(y_all[mask], probs[mask], output_dir='data/models/gnn/notebook_eval', name=split)
    print(f'-- {split} -- metrics:')
    print(json.dumps(json.loads(Path(paths['metrics']).read_text()), indent=2))

## 6. Inspecting embeddings

The 64-d embeddings extracted from the embedding head are what stage 2 (XGBoost) consumes. A quick scatter (PCA-2) of the val split, coloured by `isFraud`, gives a feel for whether the model is separating classes.

In [ ]:
from sklearn.decomposition import PCA
with torch.no_grad():
    emb = model.get_embeddings(data).cpu().numpy()
val_mask = data['transaction'].val_mask.cpu().numpy()
emb_val = emb[val_mask]
y_val = y_all[val_mask]

pca = PCA(n_components=2).fit(emb_val)
z = pca.transform(emb_val)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(z[y_val == 0, 0], z[y_val == 0, 1], s=3, alpha=0.3, label='non-fraud', color='#4c72b0')
ax.scatter(z[y_val == 1, 0], z[y_val == 1, 1], s=15, alpha=0.7, label='fraud', color='#c44e52')
ax.set_title('GNN embedding (PCA-2)')
ax.legend()
plt.tight_layout()